# Module 02 — AI Workbook Companion (Unified Memory)

A lightweight companion to [Session2_advanced.ipynb](../Session2_advanced.ipynb).
It does **not** replace the existing exercises — it adds the supervision layer
from [Module 11](../../11/README.md) to Module 02's topic: **unified memory,
page faults, prefetching, and SAXPY.**

New here? Read [Module 11's README](../../11/README.md) first — it explains the
5-phase loop (SPECIFY → GENERATE → PREDICT → VERIFY → DIAGNOSE), what a
"CPU reference" is, and the known ways AI gets CUDA wrong. This file just applies
that method to unified memory.

## The loop, applied to SAXPY on unified memory

Your task: run `c[i] = 2*a[i] + b[i]` (SAXPY) over a large array using
`cudaMallocManaged`, and make it fast with prefetching. You may have an AI write
the kernel. Your job is everything around it.

1. **SPECIFY** — Write the contract: array sizes, that memory is *unified*
   (`cudaMallocManaged`), where the data is first touched (host init vs a GPU
   init kernel), the launch configuration, and the metric (throughput, and
   whether page faults dominate).
2. **GENERATE** — Ask the AI for the SAXPY kernel and host launch. Keep its
   output in a separate file.
3. **PREDICT** — *Before running:* how many blocks/threads? Will the first run
   be dominated by **page faults** as pages migrate host→device? Will a
   `cudaMemPrefetchAsync` remove them? What correctness risks do you see in the
   index math and bounds check?
4. **VERIFY** — Use [verify_saxpy.cu](./verify_saxpy.cu): it has a CPU reference
   and a harness *stub*. You complete the launch, error check, and
   synchronization, with an element count that is **not** a multiple of the
   block size.
5. **DIAGNOSE** — Profile with `nsys`. Explain the page-fault behaviour on the
   first vs second run, and whether prefetching moved the bottleneck.

## The adversarial exercise

[adversarial_saxpy_buggy.cu](./adversarial_saxpy_buggy.cu) is a SAXPY kernel "an
AI wrote for you." It compiles and prints plausible-looking first values. **It is
wrong.** The thread-index arithmetic is subtly broken, so most output elements
are never written (and some are computed from the wrong index).

Your job:

1. Read it and predict what will happen *before* running it.
2. Build and run it. Notice that the first few printed values can look fine while
   the array as a whole is wrong.
3. State the exact bug and the one-token fix.
4. Prove your fix against a CPU reference over the **whole** array, not just the
   first five elements.

Could an AI catch this? Sometimes — but an index bug that still prints a few
correct-looking values is exactly the kind of thing a quick "looks fine" review
(human or AI) misses. A full-array check against a CPU reference catches it every
time. That check is your job.

A worked solution and explanation are in [solution.ipynb](./solution.ipynb). Try it
yourself first.

## Reflection prompt (write this down)

- The AI's kernel compiled and its first printed values looked right. What made
  it wrong, and why did printing "the first and last 5 values" fail to catch it?
- On the first launch, was the time dominated by compute or by page migration?
  What did `cudaMemPrefetchAsync` change, and why?
- SAXPY moves 3 numbers per element and does ~2 flops. Is it memory-bound or
  compute-bound, and what does that predict for the achievable speedup?


In [ ]:
# Prerequisites: verify the GPU toolchain before running this workbook.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

## Step 1 - Reproduce the bug

Build and run the problem program [adversarial_saxpy_buggy.cu](./adversarial_saxpy_buggy.cu). Read it first and predict what will happen before you run this cell.

In [ ]:
!nvcc -arch=native adversarial_saxpy_buggy.cu -o buggy && ./buggy

## Step 2 - Run your verification harness

[verify_saxpy.cu](./verify_saxpy.cu) ships a CPU reference and a PASS/FAIL gate, but it **defaults to FAIL** until you complete the `TODO` sections in it (launch, error check, synchronization). Edit the file, then re-run this cell until it prints `PASS`.

In [ ]:
!nvcc -arch=native verify_saxpy.cu -o verify && ./verify

## Next

Once your harness prints `PASS`, open **[solution.ipynb](./solution.ipynb)** to compare your fix with the worked answer and explanation. See [Module 11](../../11/README.md) for the method behind this workbook.